# model

## onnx

- onnx 文件的layer name与官方导出engine名字不同，
    - 所以我们直接从onnx导出的engine名字会继承onnx的layer name.
    - 也就导致程度读入失败core dmup
- 解决方案
    - 1. 修改onnx名字，（失败）
    - 2. 在__inspire__ 中修改outputs_layers

In [13]:
import onnx

model = onnx.load("scrfd_2.5g_bnkps.onnx")
output_names = [out.name for out in model.graph.output]
print(output_names)

['446', '466', '486', '449', '469', '489', '452', '472', '492']


- rename onnx's layer

In [ ]:
import onnx
from onnx import helper, TensorProto

model = onnx.load("scrfd_2.5g_bnkps.onnx")


rename_mapping = {
    "446": "bbox_8",
    "466": "bbox_16",
    "486": "bbox_32",
    "449": "score_8",
    "469": "score_16",
    "489": "score_32",
    "452": "kps_8",
    "472": "kps_16",
    "492": "kps_32",
}


for output in model.graph.output:
    old_name = output.name
    if old_name in rename_mapping:
        output.name = rename_mapping[old_name]

new_outputs = []
for out in model.graph.output:
    new_outputs.append(out)
model.graph.ClearField("output")
model.graph.output.extend(new_outputs)


onnx.save(model, "scrfd_2.5g_bnkps_rename.onnx")

In [15]:
import onnx

model = onnx.load("scrfd_2.5g_bnkps_rename.onnx")
output_names = [out.name for out in model.graph.output]
print(output_names)

['bbox_8', 'bbox_16', 'bbox_32', 'score_8', 'score_16', 'score_32', 'kps_8', 'kps_16', 'kps_32']


## engine

In [16]:
%%sh
export LD_LIBRARY_PATH=/kaggle/temp/InspireFace/build/inspireface-linux-tensorrt-cudaV12540_ubuntu_none/InspireFace/lib:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/opt/tensorrt/lib:$LD_LIBRARY_PATH
python engine_layer.py \
    --engine /kaggle/temp/InspireFace/test_res/pack/Megatron_TRT_/_00_scrfd_2_5g_bnkps_shape160x160_fp16

[05/13/2026-14:56:13] [TRT] [W] Using an engine plan file across different models of devices is not recommended and is likely to affect performance or even cause errors.
TensorIOMode.INPUT: input.1
TensorIOMode.OUTPUT: score_8
TensorIOMode.OUTPUT: score_16
TensorIOMode.OUTPUT: score_32
TensorIOMode.OUTPUT: bbox_8
TensorIOMode.OUTPUT: bbox_16
TensorIOMode.OUTPUT: bbox_32
TensorIOMode.OUTPUT: kps_8
TensorIOMode.OUTPUT: kps_16
TensorIOMode.OUTPUT: kps_32


In [17]:
%%sh
export LD_LIBRARY_PATH=/kaggle/temp/InspireFace/build/inspireface-linux-tensorrt-cudaV12540_ubuntu_none/InspireFace/lib:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/opt/tensorrt/lib:$LD_LIBRARY_PATH
python engine_layer.py \
    --engine _00_scrfd_2_5g_bnkps_shape160x160_fp16

TensorIOMode.INPUT: input.1
TensorIOMode.OUTPUT: 446
TensorIOMode.OUTPUT: 466
TensorIOMode.OUTPUT: 486
TensorIOMode.OUTPUT: 449
TensorIOMode.OUTPUT: 469
TensorIOMode.OUTPUT: 489
TensorIOMode.OUTPUT: 452
TensorIOMode.OUTPUT: 472
TensorIOMode.OUTPUT: 492
